# The System facade: budgeted answers, honest receipts, and named degradation

`mixle` 0.7.0 adds a thin agentic shell, `mixle.system.System`, behind three verbs:

* **`answer`** serves a query by routing to a configured *teacher* (any `generate(prompt) -> str`
  callable, or a class with `.complete(...)`), and attaches a **receipt**.
* **`ingest`** stores a produced answer as credence-weighted knowledge in a substrate store.
* **`improve`** spends a budget promoting harvested answers into an explicit captured cache.

The design commitment worth showcasing is *honesty under constraint*: spend is a hard ceiling,
every call returns an auditable receipt, and failure degrades to a **named, flagged** mode rather
than a silent guess. None of this needs a real LLM — we drive the facade with a deterministic stub
teacher so the whole notebook runs offline and reproducibly.

## 1. Answer routes to a teacher and returns a receipt

The teacher is just a callable. `answer` returns `(reply, receipt)` where the receipt records who
produced the answer, the incremental and running **spend ledger** (`frontier_calls`, `oracle_calls`,
`wall_ms`, `dollars`), the task, and whether the answer came from a captured cache.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from mixle.system import System, SystemConfig, Query

calls = {'n': 0}
def teacher(prompt):                      # any generate(prompt) -> str stands in for an LLM
    calls['n'] += 1
    return f"[teacher] {prompt.strip()}"

system = System(SystemConfig(teacher=teacher, default_budget=3))
reply, receipt = system.answer(Query("What is the capital of France?", task="qa"))

print("reply   :", reply)
print("produced_by:", receipt['produced_by'], "| task:", receipt['task'], "| captured:", receipt['captured'])
print("spend   :", receipt['spend'])
print("teacher was called", calls['n'], "time(s)")

reply   : [teacher] What is the capital of France?
produced_by: teacher | task: qa | captured: False
spend   : {'frontier_calls': 1, 'oracle_calls': 0, 'wall_ms': 0.0, 'dollars': 0.0}
teacher was called 1 time(s)


## 2. Budget is a hard ceiling, not a suggestion

A request that cannot afford the minimum-cost answer path is **refused** — the teacher is never
called, nothing is served, and the shortfall is named on the receipt. This is the property that
separates a budget from a logging field: an over-budget call produces no side effects.

In [2]:
calls['n'] = 0
reply, receipt = system.answer(Query("expensive question"), budget=0)

print("reply     :", reply)                       # None -- refused
print("status    :", receipt['status'])           # 'refused'
print("shortfall :", receipt['shortfall'])        # how far short the budget fell
print("teacher called:", calls['n'], "times")     # 0 -- no silent overspend
print("spend on a refusal:", receipt['spend'])    # all zeros

reply     : None
status    : refused
shortfall : 1.0
teacher called: 0 times
spend on a refusal: {'frontier_calls': 0, 'oracle_calls': 0, 'wall_ms': 0.0, 'dollars': 0.0}


Spend accumulates across successful calls into `system.total_spend`, and a refusal never perturbs
that running total — so the ledger stays an accurate account of what was actually spent.

In [3]:
ledger = System(SystemConfig(teacher=teacher))   # a fresh system, so the running total is unambiguous
ledger.answer(Query("a"))
ledger.answer(Query("b"), budget=0)   # refused -- must not count
ledger.answer(Query("c"))
print("total frontier_calls after [ok, refused, ok]:",
      ledger.total_spend.to_dict()['frontier_calls'])   # 2, not 3

total frontier_calls after [ok, refused, ok]: 2


## 3. Failure degrades to a *named* mode — never a silent guess

If the teacher raises, `answer` does not fabricate. With a substrate store available it falls back to
**store-only** reasoning and flags `degraded_mode='teacher_down'`; with no usable store it fails
honestly. Either way the receipt tells you exactly what happened and that no frontier call was spent.

In [4]:
from mixle.substrate.core import Substrate, SubstrateItem

def broken_teacher(prompt):
    raise ConnectionError("teacher endpoint unreachable")

store = Substrate()
store.put(SubstrateItem(kind="text", text="the rollout finished on schedule"))
degraded = System(SystemConfig(teacher=broken_teacher, store=store))

reply, receipt = degraded.answer(Query("rollout status"))
print("reply        :", reply)
print("status       :", receipt['status'])          # 'answered' -- but degraded
print("degraded_mode:", receipt['degraded_mode'])   # 'teacher_down'
print("degraded_why :", receipt['degraded_reason'])
print("produced_by  :", receipt['produced_by'], "| frontier_calls:", receipt['spend']['frontier_calls'])

reply        : [degraded: store-only] the rollout finished on schedule
status       : answered
degraded_mode: teacher_down
degraded_why : teacher endpoint unreachable
produced_by  : store | frontier_calls: 0


In [5]:
# With no store to fall back on, the same teacher failure is reported as an honest failure,
# not a hallucinated answer.
honest = System(SystemConfig(teacher=broken_teacher))
reply, receipt = honest.answer(Query("rollout status"))
print("reply :", reply, "| status:", receipt['status'])
print("reason:", receipt['reason'])

reply : None | status: failed
reason: teacher unavailable (teacher endpoint unreachable) and no usable store to fall back on


## 4. Ingest: store an answer as credence-weighted knowledge

`ingest` writes a produced answer through the store boundary as one or more claims. With no store it
is an honest no-op (`status='no_store'`); with a substrate store the claim is assimilated and counted.

In [6]:
knower = System(SystemConfig(teacher=teacher, store=Substrate()))
report = knower.ingest("the sky is blue", source={"model": "teacher-v1"})
print("ingest status:", report['status'], "| claims assimilated:", report['n_claims'])

# improve() is the third verb -- it spends a budget promoting harvested answers into a captured
# cache. Without an improvement subsystem registered it reports that honestly rather than pretending.
print("improve      :", knower.improve(budget=2))

ingest status: ok | claims assimilated: 1
improve      : {'status': 'nothing_to_improve', 'reason': 'no improvement subsystem registered yet', 'budget': 2}


## What 0.7.0 gives you here

The `System` facade is deliberately thin, and every verb is *auditable*:

* **`answer`** attaches a receipt with an explicit spend ledger; **budget is a hard ceiling** enforced
  before any teacher call, so an over-budget request has no side effects.
* **Degradation is named and flagged** (`teacher_down`, `store_down`) — the system never trades a
  failure for a silent guess.
* **`ingest`** and **`improve`** move produced answers into credence-weighted storage and a captured
  cache, so measured savings are attributable to explicit improvement rather than implicit caching.

Swap the stub `teacher` for an `OpenAICompatLLM` (also in `mixle.system`) and the same three verbs,
the same receipts, and the same guarantees drive a real endpoint.